In [ ]:
# we assess accuracy of raw gridded temperature products using valid insitu stations

# FIX__________ for validation use only - correlation, nrmse and rsd

import xarray
import pandas as pd 
import glob
import os
import rasterio
import concurrent.futures
from scipy.stats import spearmanr
import numpy as np  
from permetrics.regression import RegressionMetric
import HydroErr as he   
import datetime as dt   

datasets = ['ERA5L', 'HAR', 'CHIRTS']

dataset_base_dir = r"..\Data_Temp\Datasets"
output_dir = fr"..\Data_Temp\Processed\Initial_accuracy"
os.makedirs(output_dir, exist_ok= True)

In [ ]:
# Read Insitu Data

insitu_tmax_df = pd.read_parquet(r"..\Data_Temp\Insitu\Cleaned\tmax_cleaned.parquet")
insitu_tmin_df = pd.read_parquet(r"..\Data_Temp\Insitu\Cleaned\tmin_cleaned.parquet")
insitu_tmax_df['date'] = pd.to_datetime(insitu_tmax_df[['year', 'month', 'day']])
insitu_tmin_df['date'] = pd.to_datetime(insitu_tmin_df[['year', 'month', 'day']])

#
unique_stn_tmax = insitu_tmax_df[['index_no', 'lat', 'long']].drop_duplicates()
unique_stn_tmin = insitu_tmin_df[['index_no', 'lat', 'long']].drop_duplicates()
stn_points_tmax = list(zip(unique_stn_tmax['long'], unique_stn_tmax['lat']))
stn_points_tmin = list(zip(unique_stn_tmin['long'], unique_stn_tmin['lat']))

unique_stn_tmax.to_csv(os.path.join(output_dir, "unique_stn_tmax.csv"), index = False)
unique_stn_tmin.to_csv(os.path.join(output_dir, "unique_stn_tmin.csv"), index = False)

,date,index_no,year,month,day,max_temp
0,2001-01-01,104,2001,1,1,12.7
1,2001-01-02,104,2001,1,2,7.7
2,2001-01-03,104,2001,1,3,14.0
3,2001-01-04,104,2001,1,4,14.2
4,2001-01-05,104,2001,1,5,14.2


In [3]:
#Get path, files and file list for datasets
# address tmax tmin situation

def extract_date(file, dataset):
    filename = os.path.basename(file)
    date_str = filename[-12:-4]
    return date_str

def get_dataset_info(dataset):
    dataset_path = fr"..\Data_Temp\Datasets\{dataset}"
    dataset_tmax_files = glob.glob(os.path.join(dataset_path, f"{dataset}_tmax_*.tif"))
    dataset_tmin_files = glob.glob(os.path.join(dataset_path, f"{dataset}_tmin_*.tif"))
    dataset_dates = list(map(lambda file: extract_date(file, dataset), dataset_tmax_files))
    return dataset_path, dataset_tmax_files, dataset_tmin_files, dataset_dates

dataset_results = {}
for dataset in datasets:
    dataset_path, dataset_tmax_files, dataset_tmin_files, dataset_dates = get_dataset_info(dataset)
    dataset_results[f'{dataset}_path'] = dataset_path
    dataset_results[f'{dataset}_tmax_files'] = dataset_tmax_files
    dataset_results[f'{dataset}_tmin_files'] = dataset_tmin_files
    dataset_results[f'{dataset}_dates'] = dataset_dates

#dataset_results['MSWEP_files']

In [ ]:
def extract_concurrent_grid_data(dataset_results, datasets, stn_points_tmax, unique_stn_tmax, stn_points_tmin, unique_stn_tmin):
    df_final_tmax = None
    df_final_tmin = None
    
    # Core function to process a single file
    def process_file(file, dataset, stn_points, unique_stn):
        date_obj = extract_date(file, dataset) 
        local_records = []
        try:
            with rasterio.open(file) as src:
                crs = src.crs
                if crs == "EPSG:4326":
                    sampled_values = list(src.sample(stn_points))
                    for i, value in enumerate(sampled_values):
                        local_records.append({
                            'date': date_obj,
                            'stn_id': unique_stn.iloc[i]['index_no'],
                            'temp_grid': value[0]
                        })
        except Exception as e:
            print(f'Error processing {file}: {e}')
        return local_records

    # Helper function to run the thread pool for a given set of files (tmax or tmin)
    def process_variable(files, var_name, dataset, stn_points, unique_stn):
        dataset_records = []
        with concurrent.futures.ThreadPoolExecutor() as executor:
            # Pass the dataset argument alongside the file
            results = executor.map(lambda f: process_file(f, dataset, stn_points, unique_stn), files)
            for result in results:
                dataset_records.extend(result)
        
        df = pd.DataFrame(dataset_records)
        if not df.empty:
            # Filter out standard NoData values rather than all negative temperatures
            df = df[df['temp_grid'] != -9999.0] 
            df = df.rename(columns={'temp_grid': f'{dataset}_{var_name}'})
        return df

    # Loop through each dataset and process both variables
    for dataset in datasets:
        tmax_files = dataset_results[f'{dataset}_tmax_files']
        tmin_files = dataset_results[f'{dataset}_tmin_files']

        print(f"Processing {dataset}...")
        
        # --- Process TMAX ---
        print(f"  Extracting TMAX ({len(tmax_files)} files)...")
        df_tmax = process_variable(tmax_files, 'tmax', dataset, stn_points_tmax, unique_stn_tmax)
        
        if not df_tmax.empty:
            if df_final_tmax is None:
                df_final_tmax = df_tmax
            else:
                df_final_tmax = pd.merge(df_final_tmax, df_tmax, on=['date', 'stn_id'], how='inner')
                print(f"  Merged {dataset} TMAX. Shape: {df_final_tmax.shape}")
        else:
            print(f"  Warning: No data extracted for TMAX {dataset}")

        # --- Process TMIN ---
        print(f"  Extracting TMIN ({len(tmin_files)} files)...")
        df_tmin = process_variable(tmin_files, 'tmin', dataset, stn_points_tmin, unique_stn_tmin)
        
        if not df_tmin.empty:
            if df_final_tmin is None:
                df_final_tmin = df_tmin
            else:
                df_final_tmin = pd.merge(df_final_tmin, df_tmin, on=['date', 'stn_id'], how='inner')
                print(f"  Merged {dataset} TMIN. Shape: {df_final_tmin.shape}")
        else:
            print(f"  Warning: No data extracted for TMIN {dataset}")

    return df_final_tmax, df_final_tmin

# Run the extraction for both variables
df_final_tmax, df_final_tmin = extract_concurrent_grid_data(
    dataset_results, 
    datasets, 
    stn_points_tmax, 
    unique_stn_tmax, 
    stn_points_tmin, 
    unique_stn_tmin
)

print("Extraction complete.")
print(f"TMAX Data Shape: {df_final_tmax.shape if df_final_tmax is not None else 'None'}")
print(f"TMIN Data Shape: {df_final_tmin.shape if df_final_tmin is not None else 'None'}")

Processing ERA5L...
  Extracting TMAX (1826 files)...
  Extracting TMIN (1826 files)...
Processing HAR...
  Extracting TMAX (1826 files)...
  Merged HAR TMAX. Shape: (109560, 4)
  Extracting TMIN (1826 files)...
  Merged HAR TMIN. Shape: (109560, 4)
Processing CHIRTS...
  Extracting TMAX (1826 files)...
  Merged CHIRTS TMAX. Shape: (109560, 5)
  Extracting TMIN (1826 files)...
  Merged CHIRTS TMIN. Shape: (109560, 5)
Extraction complete.
TMAX Data Shape: (109560, 5)
TMIN Data Shape: (109560, 5)


In [5]:
df_insitu_tmax = insitu_tmax_df[['date', 'index_no', 'max_temp']]
df_insitu_tmin = insitu_tmin_df[['date', 'index_no', 'min_temp']]

df_insitu_tmax = df_insitu_tmax.rename(columns={'max_temp':'INSITU_tmax', 'index_no':'stn_id'})
df_insitu_tmin = df_insitu_tmin.rename(columns={'min_temp':'INSITU_tmin', 'index_no':'stn_id'})

df_final_tmax['date'] = pd.to_datetime(df_final_tmax['date'])
df_final_tmin['date'] = pd.to_datetime(df_final_tmin['date'])

df_insitu_tmax['date'] = pd.to_datetime(df_insitu_tmax['date'])
df_insitu_tmin['date'] = pd.to_datetime(df_insitu_tmin['date'])

df_merged_tmax = pd.merge(df_final_tmax, df_insitu_tmax, on = ['date', 'stn_id'], how = 'inner')
df_merged_tmin = pd.merge(df_final_tmin, df_insitu_tmin, on = ['date', 'stn_id'], how = 'outer')
df_merged_tmax.to_csv(os.path.join(output_dir, r"merged_tmax.csv"), index = False)
df_merged_tmin.to_csv(os.path.join(output_dir, r"merged_tmin.csv"), index = False)

## CALCULATING ERROR METRICES - BREAKPOINT ##

In [6]:
#checkpoint - run from here to reduce code runtime while rerunning
import xarray
import pandas as pd
import glob
import os
import rasterio
import concurrent.futures
from scipy.stats import spearmanr
import os
import numpy as np
from permetrics.regression import RegressionMetric
import HydroErr as he
import datetime as dt

datasets = ['ERA5L', 'HAR', 'CHIRTS']
output_dir = fr"..\Data_Temp\Processed\Initial_accuracy"

df_merged_tmax = pd.read_csv(os.path.join(output_dir, r"merged_tmax.csv"))
df_merged_tmin = pd.read_csv(os.path.join(output_dir, r"merged_tmin.csv"))

In [7]:
#column normalization function
def normalize_column(col):
    return (col - col.min()) / (col.max() - col.min())
          
#calculate accuracy of datasets
def calculate_accuracy(dataframe_temp: pd.DataFrame, t_variable: str):
    df_accuracy = pd.DataFrame()

    for dataset in datasets:
        dataset_accuracy = {'dataset': dataset}

        #Convert to array
        temp_grid_array = dataframe_temp[f'{dataset}_{t_variable}'].values
        temp_insitu_array = dataframe_temp[f'INSITU_{t_variable}'].values
    
        #drop null values, and negative values
        valid_indices = np.logical_and.reduce([
            ~np.isnan(temp_grid_array),
            ~np.isnan(temp_insitu_array),
            temp_grid_array >=  0,
            temp_insitu_array >= 0
        ])
    
        observations = np.sum(valid_indices)
        dataset_accuracy['observations_n'] = observations

        temp_grid_array = temp_grid_array[valid_indices]
        temp_insitu_array = temp_insitu_array[valid_indices]

        #correlation coefficient
        correlation = he.spearman_r(temp_grid_array, temp_insitu_array)
        dataset_accuracy['correlation'] = correlation

        #NRMSE
        nrmse = he.nrmse_mean(temp_grid_array, temp_insitu_array)
        dataset_accuracy['nrmse'] = nrmse

        #Percent bias
        pbias = 100 * np.sum(temp_grid_array - temp_insitu_array) / np.sum(temp_insitu_array)
        dataset_accuracy['pbias'] = pbias

        #Relative standard deviation 
        rsd = (np.std(temp_grid_array))/(np.std(temp_insitu_array))
        dataset_accuracy['rsd'] = rsd

        df_dataset_accuracy = pd.DataFrame([dataset_accuracy])
        df_accuracy = pd.concat([df_accuracy, df_dataset_accuracy], ignore_index=True)

    #Normalization
    df_accuracy['abs_pbias'] = df_accuracy['pbias'].abs()
    df_accuracy['diff_rsd'] = (df_accuracy['rsd']-1).abs()

    df_accuracy_normalized = df_accuracy.copy().drop(columns=['pbias'])

    metrics = ['correlation', 'nrmse', 'abs_pbias', 'diff_rsd']
    for column in metrics:
        df_accuracy_normalized[column] = normalize_column(df_accuracy_normalized[column])

    #converting to scale of 0 to 1 (ideal to worst) - correlation & rsd needs inversion
    df_accuracy_normalized['correlation'] = 1 - df_accuracy_normalized['correlation']

    #get cpi for each (linear)
    df_accuracy_normalized['cpi'] = df_accuracy_normalized['correlation'] + df_accuracy_normalized['nrmse'] + df_accuracy_normalized['abs_pbias'] + df_accuracy_normalized['diff_rsd']
    df_accuracy_normalized['cpi_rank'] = df_accuracy_normalized['cpi'].rank(ascending=True)

    df_accuracy_final = df_accuracy_normalized[['dataset','cpi','cpi_rank']].merge(df_accuracy[['dataset','correlation','nrmse','pbias','rsd', 'observations_n']], on=['dataset'], how='inner')
    return df_accuracy_final

#get daily data accuracy
daily_accuracy_tmax = calculate_accuracy(df_merged_tmax, 'tmax')
daily_accuracy_tmin = calculate_accuracy(df_merged_tmin, 'tmin')

print(daily_accuracy_tmax)
print(daily_accuracy_tmin)

  dataset       cpi  cpi_rank  correlation     nrmse      pbias       rsd  \
0   ERA5L  3.283606       2.0     0.820779  0.212064 -15.376199  1.076486   
1     HAR  3.433446       3.0     0.846039  0.196387 -13.408894  1.140289   
2  CHIRTS  0.000000       1.0     0.906762  0.102065  -0.272916  0.948772   

   observations_n  
0          102371  
1          102821  
2          103320  
  dataset       cpi  cpi_rank  correlation     nrmse     pbias       rsd  \
0   ERA5L  2.760276       3.0     0.870638  0.229111 -7.479053  0.994373   
1     HAR  0.626852       1.0     0.900477  0.195079 -1.155719  1.007756   
2  CHIRTS  2.177072       2.0     0.927503  0.201105  9.472881  1.019674   

   observations_n  
0           95250  
1           97922  
2           98665  


In [8]:
# Monthly scale comparison

def calculate_monthly_accuracy(df_temp: pd.DataFrame, t_variable: str):
    # Ensure date is datetime
    df_temp = df_temp.copy()
    df_temp['date'] = pd.to_datetime(df_temp['date'])
    df_temp['year'] = df_temp['date'].dt.year
    df_temp['month'] = df_temp['date'].dt.month


    cols_to_agg = [f'ERA5L_{t_variable}', f'HAR_{t_variable}', f'CHIRTS_{t_variable}', f'INSITU_{t_variable}']
    
    df_monthly = df_temp.groupby(by=['year', 'month'])[cols_to_agg].mean().reset_index()

    df_accuracy_monthly = calculate_accuracy(df_monthly, t_variable)
    
    return df_accuracy_monthly

monthly_accuracy_tmax = calculate_monthly_accuracy(df_merged_tmax, "tmax")
monthly_accuracy_tmin = calculate_monthly_accuracy(df_merged_tmin, "tmin")

print("MONTHLY ACCURACY TMAX")
print(monthly_accuracy_tmax)
print("\nMONTHLY ACCURACY TMIN")
print(monthly_accuracy_tmin)


MONTHLY ACCURACY TMAX
  dataset       cpi  cpi_rank  correlation     nrmse      pbias       rsd  \
0   ERA5L  3.000000       3.0     0.992776  0.157408 -15.657317  0.948199   
1     HAR  2.377865       2.0     0.983884  0.134813 -13.317834  1.022617   
2  CHIRTS  1.000000       1.0     0.966435  0.019358  -0.148964  1.006567   

   observations_n  
0              60  
1              60  
2              60  

MONTHLY ACCURACY TMIN
  dataset       cpi  cpi_rank  correlation     nrmse      pbias       rsd  \
0   ERA5L  2.000000       2.0     0.990664  0.109627 -10.211878  0.998461   
1     HAR  0.835932       1.0     0.990386  0.038698  -1.400689  0.969202   
2  CHIRTS  3.973906       3.0     0.989775  0.108402  10.134186  1.057436   

   observations_n  
0              60  
1              60  
2              60  


In [ ]:
# error correlation: do CHIRTS and ERA5L miss on the same days/stations,
# or do they fail independently? (uses the merged dataframe you already have)
df_merged_tmax['CHIRTS_error'] = df_merged_tmax['CHIRTS_tmax'] - df_merged_tmax['INSITU_tmax']
df_merged_tmax['ERA5L_error']  = df_merged_tmax['ERA5L_tmax']  - df_merged_tmax['INSITU_tmax']

error_corr = df_merged_tmax[['CHIRTS_error', 'ERA5L_error']].corr().iloc[0, 1]
print(f"Error correlation (Tmax): {error_corr:.3f}")